# NB-003_regression_gates.ipynb — Mod 03 Regression Gates (notebook-first prototype, D2)

Mod 03 — Regression Gates: notebook-first prototype (D2), promotes into `src/llmops/eval/`.

**Links** — companion LLD: `doc/design/03_lld_tests.md`; task 03 contract: `doc/task/03_regression_gates.md`; step4 parity reference: `data/docs/s4_qa_deep_dives.md:487,500-502` (regression & quality gates row + Interview Q2).

**Scope banner** — offline-deterministic only; live/judge path is OUT (D5); Mod 4 owns latency computation.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import sys
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal, TypeAlias
from dotenv import load_dotenv

In [2]:
# --- Repo root resolution ---------------------------------------------
# Makes the relative "data/..." and "eval/..." paths work no matter
# where Jupyter was launched from.
ROOT = Path.cwd().resolve()

while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError(
        f"pyproject.toml not found while walking upward from {Path.cwd()}"
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


assert (ROOT / "pyproject.toml").exists(), (
    f"pyproject.toml not found in any parent directory — stopped at {ROOT}"
)

print("repo root:", ROOT)

repo root: /home/dipak/agentic/step9_llmops


In [3]:
GOLDENS_DIR = ROOT / "eval" / "goldens"
DOCS_DIR = ROOT / "data" / "docs"
BASELINES_DIR = ROOT / "eval" / "baselines"

print(f"ROOT   = {ROOT}")
print(f"GOLDENS_DIR = {GOLDENS_DIR}  exists={GOLDENS_DIR.exists()}")
print(f"DOCS_DIR    = {DOCS_DIR}  exists={DOCS_DIR.exists()}")
print(f"BASELINES_DIR = {BASELINES_DIR}  exists={BASELINES_DIR.exists()}")

ROOT   = /home/dipak/agentic/step9_llmops
GOLDENS_DIR = /home/dipak/agentic/step9_llmops/eval/goldens  exists=True
DOCS_DIR    = /home/dipak/agentic/step9_llmops/data/docs  exists=True
BASELINES_DIR = /home/dipak/agentic/step9_llmops/eval/baselines  exists=True


**Offline seam note** — this notebook must never read `.env` / `LLM_*` / `GROQ_*` — offline seam (D5/D6, T-03-11).

## Section 1 — `metric_registry` prototyping

Metric schema is DATA, not code — `Metric` dataclass, `MetricKind`/`MetricDirection` TypeAliases.

In [4]:
# --- Offline-seam self-check (belongs here, not just Section 9) ---
# Prove at setup time that nothing in this kernel session has pulled in
# LIVE MODEL / JUDGE code, before a single eval line runs.

# Why NO env-presence assert: .env (gitignored) injects LLM_*/GROQ_*/GEMINI_*/
# EMBED_MODEL/LANGSMITH_* into the DEV PROCESS ENV by design (task 00b:82).
# Their presence in os.environ is normal, not leakage. The offline contract
# (T-03-11/T-03-11b) is that the EVAL CODE never READS env — proven by the
# env-spy (shim-reject on os.environ/os.getenv/.env read) in tests/test_gates.py,
# not by a presence assert on a dev machine.

# Why NO socket/urllib/http.client deny-assert: those stdlib networking
# modules are imported by IPython/Jupyter itself (ZMQ kernel, cell I/O).
# Their presence in sys.modules is kernel infrastructure, not eval leakage.
# The authoritative offline proof is the CLOSURE scan over the eval modules
# (T-03-11) + env-spy (T-03-11b) in tests/test_gates.py — not kernel-wide
# sys.modules. Here we only flag genuine live-model/judge modules.

_DENY_MODULES = {
    "config.judge", "judge_llm", "openai", "langchain", "langchain_openai",
}
_loaded = _DENY_MODULES & set(sys.modules)
assert not _loaded, f"Live model/judge module already imported: {_loaded}"

print("Offline seam OK: no live model / judge modules imported at setup.")
print("Env vars ARE present in the dev process (normal). Read-side and closure")
print("enforcement => T-03-11/T-03-11b in tests/test_gates.py.")

Offline seam OK: no live model / judge modules imported at setup.
Env vars ARE present in the dev process (normal). Read-side and closure
enforcement => T-03-11/T-03-11b in tests/test_gates.py.


**Tolerance policy** — judge-default ±0.03 absolute vs per-source 1/(n+1) single-flip floor vs latency ±0.20 relative (D10/D19).

In [5]:
# --- Metric dataclass --------------------------------------------------
# Registry is DATA ONLY — no evaluator functions live here (LLD:
# "registry is DATA"; 03:84). run_suite (Section 2) computes values; this
# table only declares identity, tolerance policy, and provenance.

MetricKind: TypeAlias = Literal["gate", "guardrail", "info"]
MetricDirection: TypeAlias = Literal["higher", "lower"]  # bigger = better

@dataclass(frozen=True)
class Metric:
    id: str                     # dotted id, step4-parity naming (T-03-1 regex)
    name: str                   # human-readable
    kind: MetricKind            # gate | guardrail | info
    direction: MetricDirection
    tolerance: float            # absolute delta (0.03 judge) or relative fraction (0.20 latency)
    tolerance_unit: Literal["absolute", "relative"]
    value_domain: Literal["fraction", "nonnegative"]  # fraction = finite ∩ [0,1]
    expected_sample_size: int | None  # committed closed-set n (D19); None for non-n rows
    description: str            # one-line semantics. NO evaluator function here.

# Per-source committed golden counts (D19 closed set, verified against
# eval/goldens/*.json) — the single source of truth for tolerance floors.
RETRIEVER_N = {"S1": 34, "S2": 27, "S3": 28, "S4": 28, "S5": 26}
CORRECTNESS_N = {"S1": 25, "S2": 22, "S3": 20, "S4": 24, "S5": 21}
MISROUTE_N = {"S1": 1, "S2": 2, "S3": 2, "S4": 2, "S5": 1}

def _single_flip_tol(n: int) -> float:
    """Tolerance floor s.t. a single flipped golden row always trips
    the gate: (n-1)/n must fall strictly below n/(n+1). D19/D39."""
    return 1.0 / (n + 1)

REGISTRY: tuple[Metric, ...] = (
    Metric(
        id="eval.gate.golden_rules",
        name="Golden files pass L1 invariants",
        kind="gate", direction="higher",
        tolerance=0.0, tolerance_unit="absolute",
        value_domain="fraction",
        expected_sample_size=3,   # 3 golden FILES, not rows (T-03-2b)
        description="fraction of golden files passing L1 invariant checks "
                     "(reuses tools/goldens/t01_verify.py) — global, all 3 files",
    ),
    *(
        Metric(
            id=f"eval.gate.retriever.agreement.{s}",
            name=f"Retriever agreement — {s}",
            kind="gate", direction="higher",
            tolerance=_single_flip_tol(n), tolerance_unit="absolute",
            value_domain="fraction",
            expected_sample_size=n,
            description=f"per-source ideal_context ↔ must_contain coverage ({s}, n={n})",
        )
        for s, n in RETRIEVER_N.items()
    ),
    *(
        Metric(
            id=f"eval.gate.routing.misroute_negation.{s}",
            name=f"Misroute negation — {s}",
            kind="gate", direction="higher",
            tolerance=0.03, tolerance_unit="absolute",   # judge-default; n=1/2 already trips on any flip
            value_domain="fraction",
            expected_sample_size=n,
            description=f"misroute-category rows stay true negatives ({s}, n={n})",
        )
        for s, n in MISROUTE_N.items()
    ),
    *(
        Metric(
            id=f"eval.gate.correctness.answer_cited.{s}",
            name=f"Answer citation coverage — {s}",
            kind="gate", direction="higher",
            tolerance=_single_flip_tol(n), tolerance_unit="absolute",
            value_domain="fraction",
            expected_sample_size=n,
            description=f"ideal_answer covers every citation-critical must_contain string ({s}, n={n})",
        )
        for s, n in CORRECTNESS_N.items()
    ),
    Metric(
        id="eval.guardrail.calibration.band8",
        name="Judge calibration — band ≥8 share",
        kind="guardrail", direction="higher",
        tolerance=0.03, tolerance_unit="absolute",
        value_domain="fraction",
        expected_sample_size=None,   # nightly live only (D29)
        description="share of flawless/grounded rows scoring ≥8 under the judge — nightly live only",
    ),
    Metric(
        id="eval.guardrail.position_bias.agreement",
        name="Judge position-bias agreement",
        kind="guardrail", direction="higher",
        tolerance=0.03, tolerance_unit="absolute",
        value_domain="fraction",
        expected_sample_size=None,   # nightly live only (D28)
        description="judge A/B vs B/A agreement on the same row — nightly live only",
    ),
    *(
        Metric(
            id=f"eval.info.snapshot_rowcount.{s}",
            name=f"Golden rows evaluated — {s}",
            kind="info", direction="higher",
            tolerance=0.0, tolerance_unit="absolute",
            value_domain="nonnegative",
            expected_sample_size=None,
            description=f"provenance only, never verdict ({s})",
        )
        for s in ("S1", "S2", "S3", "S4", "S5")
    ),
    Metric(
        id="eval.guardrail.latency.p95",
        name="P95 total latency",
        kind="guardrail", direction="lower",
        tolerance=0.20, tolerance_unit="relative",
        value_domain="nonnegative",
        expected_sample_size=None,   # Mod-4 computed (task 04 SLO report)
        description="P95 total latency vs baseline*1.2 — registered now, value arrives with Mod 4",
    ),
    Metric(
        id="eval.info.latency.ttft_p95",
        name="TTFT P95",
        kind="info", direction="lower",
        tolerance=0.20, tolerance_unit="relative",
        value_domain="nonnegative",
        expected_sample_size=None,   # Mod-4 computed
        description="TTFT P95, recorded for provenance only — Mod-4 computed",
    ),
)

print(f"Registered {len(REGISTRY)} metric rows "
      f"({sum(1 for m in REGISTRY if m.kind == 'gate')} gate, "
      f"{sum(1 for m in REGISTRY if m.kind == 'guardrail')} guardrail, "
      f"{sum(1 for m in REGISTRY if m.kind == 'info')} info)")

Registered 25 metric rows (16 gate, 3 guardrail, 6 info)


**Reconciliation note** — 139 (retriever) + 8 (misroute) + 107 (correctness) + 59 (ungated categories) = 313 — verified against golden files.

In [6]:
# --- Accessors (registry-declaration order preserved) -------------------

def get_metric(metric_id: str) -> Metric:
    """Raise KeyError on unknown id (T-03-10)."""
    for m in REGISTRY:
        if m.id == metric_id:
            return m
    raise KeyError(f"unknown metric id: {metric_id}")

def metric_ids() -> list[str]:
    """Registry-declaration order (asserted by T-03-1)."""
    return [m.id for m in REGISTRY]

def gate_metric_ids() -> tuple[str, ...]:
    """Gates only, declaration order; len == 16 (T-03-3)."""
    return tuple(m.id for m in REGISTRY if m.kind == "gate")

In [7]:
# --- validate_registry() ------------------------------------------------
# Invariant checks (T-03-1). NOTE the scoping (LLD 03:89 vs 03:103/113):
# the 1/(n+1) single-flip floor applies to the retriever.agreement and
# correctness.answer_cited per-source families ONLY. misroute gates keep
# the judge ±0.03 default by design (n=1/2 already trips on any flip);
# a literal "every per-source gate row" floor read WOULD reject them.

import re as _re

_ID_RE = _re.compile(
    r"^eval\.(gate|guardrail|info)\.[a-z_]+(\.[a-z0-9_]+)*(\.S[1-5])?$"
)

_FLOOR_FAMILIES = ("eval.gate.retriever.agreement.", "eval.gate.correctness.answer_cited.")

def validate_registry() -> None:
    """Raise AssertionError listing every violation (T-03-1)."""
    errors: list[str] = []
    seen: set[str] = set()

    for m in REGISTRY:
        if m.id in seen:
            errors.append(f"duplicate id: {m.id}")
        seen.add(m.id)

        if not _ID_RE.fullmatch(m.id):
            errors.append(f"id does not match regex: {m.id}")

        if m.kind not in ("gate", "guardrail", "info"):
            errors.append(f"bad kind: {m.id}={m.kind!r}")
        if m.direction not in ("higher", "lower"):
            errors.append(f"bad direction: {m.id}={m.direction!r}")
        if m.value_domain not in ("fraction", "nonnegative"):
            errors.append(f"bad value_domain: {m.id}={m.value_domain!r}")
        if not (0.0 <= m.tolerance < float("inf")):
            errors.append(f"tolerance out of range: {m.id} tol={m.tolerance}")

        is_floor = m.kind == "gate" and m.id.startswith(_FLOOR_FAMILIES)
        is_misroute = m.kind == "gate" and m.id.startswith("eval.gate.routing.misroute_negation.")

        if is_floor or is_misroute:
            if m.expected_sample_size is None or m.expected_sample_size < 1:
                errors.append(f"per-source gate missing expected_sample_size: {m.id}")
            elif is_floor:
                expect = 1.0 / (m.expected_sample_size + 1)
                if abs(m.tolerance - expect) > 1e-12:
                    errors.append(
                        f"floor-family tolerance != 1/(n+1): {m.id} "
                        f"tol={m.tolerance} != {expect}"
                    )

    if not errors:
        return
    raise AssertionError("validate_registry failed:\n  - " + "\n  - ".join(errors))

validate_registry()
print("validate_registry() OK —", len(REGISTRY), "rows, 16 gates, floors scoped to "
      "retriever.agreement + correctness.answer_cited (misroute keeps 0.03).")

validate_registry() OK — 25 rows, 16 gates, floors scoped to retriever.agreement + correctness.answer_cited (misroute keeps 0.03).


Inline sanity-check narrative → promotes to T-03-1.

## Section 2 — `run_suite` prototyping

Offline deterministic evaluator, pure function of committed goldens (+ corpus reads
in exactly the LLD's sanctioned ways: `corpus_sha256` manifest, `golden_rules` L1 via
the imported `t01_verify` module, and the per-source stubs' source-bundle match seam —
LLD 03:157-178). No free-corpus/retriever/answer search.

**Explicit non-goals** — no network, no env reads, no `config/judge.py` import (D5/D6, offline seam).

In [8]:
#Input Validation Error & Text Normalization
# Custom exception for evaluation input/file validation errors (Exit Code 3)
class EvaluationInputError(Exception):
    """Raised when golden datasets, corpus paths, or input structures are missing/malformed."""
    pass


def _normalize_text(text: str) -> str:
    """NFC normalization, line ending standardization (\r\n -> \n), and edge whitespace trimming."""
    if not isinstance(text, str):
        return ""
    normalized = unicodedata.normalize("NFC", text)
    normalized = normalized.replace("\r\n", "\n").replace("\r", "\n")
    return normalized.strip()

In [9]:
#Corpus Deterministic SHA-256 Digest

def compute_corpus_sha256(docs_dir: Path = DOCS_DIR) -> str:
    """Canonical manifest: sorted RELATIVE paths + file bytes (LLD 03:157-158)."""
    if not docs_dir.exists() or not docs_dir.is_dir():
        raise EvaluationInputError(f"Docs directory not found: {docs_dir}")

    rel_files = sorted(
        f.relative_to(docs_dir) for f in docs_dir.rglob("*") if f.is_file()
    )
    if not rel_files:
        raise EvaluationInputError(f"No document files found in docs directory: {docs_dir}")

    hasher = hashlib.sha256()
    for rel in rel_files:
        hasher.update(str(rel).encode("utf-8"))   # relative path, not basename
        hasher.update(b"\x00")
        hasher.update((docs_dir / rel).read_bytes())
    return hasher.hexdigest()

**Normalization contract** — NFC, line-ending normalize, trim, case-preserved substring match — owned by `t01_verify` helper, imported (not subprocess).

In [10]:
#Golden File L1 Invariant Checker
# Reuse tools/goldens/t01_verify.py IMPORTED as a module (T-03-11 closure
# seam, LLD 03:159-161). verify(path, lo, hi) -> (rows, okflag).
# Wide lo/hi (1..10**9) keeps the count-range check quiet; okflag tests all
# L1 invariants (T-01-1..7) + file-level targets. NO weak inline fallback.

import tools.goldens.t01_verify as _t01

def verify_golden_file_invariants(file_path: Path) -> bool:
    """True iff the file passes t01_verify's L1 invariant + file-level checks."""
    _rows, okflag = _t01.verify(str(file_path), 1, 10**9)
    return bool(okflag)

In [11]:
#Evaluator Logic Across Golden Categories
# Per-file -> metric-family mapping (D15, LLD 03:121-127):
#   retriever_goldens.json           -> eval.gate.retriever.agreement.{S}
#   correctness_goldens.json         -> eval.gate.correctness.answer_cited.{S}
#   query_processing misroute rows   -> eval.gate.routing.misroute_negation.{S}
# STUB PREDICATE (LLD 03:176-178 + T-03-4): a golden's must_contain members are
# matched against the NORMALIZED, JOINED SOURCE-BUNDLE text (groundedness per
# T-01-3), NOT against ideal_context/ideal_answer themselves — literal coverage
# of those fields reads 0.50-0.81 on committed goldens and would break T-03-4's
# "every gate row == 1.0 on committed goldens". Matching reuses the imported
# t01_verify module's corpus read (_t01.bund) — the sanctioned closure seam.
# No free-corpus / retriever / answer search anywhere (LLD 03:162-163).

SOURCE_IDS = ("S1", "S2", "S3", "S4", "S5")

def _split_sources(row) -> list[str]:
    """Split-on-`+` (D15): a multi-source union row counts once per listed source."""
    return [s.strip() for s in str(row.get("source", "")).split("+") if s.strip()]

def _grounded(row) -> bool:
    """all must_contain present in the row's normalized joined source-bundle text."""
    bundles = [ _normalize_text(_t01.bund[s]) for s in _split_sources(row) if s in _t01.bund ]
    joined = "\n".join(bundles)
    return all(_normalize_text(mc) in joined for mc in row.get("must_contain", []))

def _per_source_hits(rows) -> dict[str, tuple[int, int]]:
    """{source: (grounded_rows, n)} using the normalized substring contract."""
    out = {s: [0, 0] for s in SOURCE_IDS}
    for r in rows:
        mcs = r.get("must_contain")
        if not isinstance(mcs, list) or not mcs or not all(
                isinstance(mc, str) and mc for mc in mcs):
            raise EvaluationInputError(
                f"Empty/invalid must_contain at eval time: {r.get('id','?')} (H8)")
        grounded = _grounded(r)
        for s in _split_sources(r):
            if s in out:
                out[s][1] += 1
                if grounded:
                    out[s][0] += 1
    return out

def evaluate_golden_records(goldens_dir: Path) -> dict[str, tuple[float, int]]:
    """Compute (value, sample_size) per computed metric id (offline, deterministic)."""
    if not goldens_dir.exists() or not goldens_dir.is_dir():
        raise EvaluationInputError(f"Goldens directory not found: {goldens_dir}")

    golden_files = sorted(goldens_dir.glob("*_goldens.json"))
    if not golden_files:
        raise EvaluationInputError(f"No golden files found in: {goldens_dir}")

    files: dict[str, list[dict]] = {}
    for fp in golden_files:
        try:
            files[fp.name] = json.loads(fp.read_text(encoding="utf-8"))
        except json.JSONDecodeError as e:
            raise EvaluationInputError(f"Malformed golden JSON in {fp.name}: {e}") from e

    values: dict[str, tuple[float, int]] = {}

    # golden_rules — real L1 via t01_verify, 3-file denominator (T-03-2b)
    passing = sum(1 for fp in golden_files if verify_golden_file_invariants(fp))
    values["eval.gate.golden_rules"] = (passing / len(golden_files), len(golden_files))

    # retriever.agreement — ONLY retriever_goldens.json rows, per source
    for s, (hits, n) in _per_source_hits(
            files.get("retriever_goldens.json", [])).items():
        if n == 0:
            raise EvaluationInputError(f"Zero retriever rows for {s} (never a 0.0 gate)")
        values[f"eval.gate.retriever.agreement.{s}"] = (hits / n, n)

    # correctness.answer_cited — ONLY correctness_goldens.json rows, per source
    for s, (hits, n) in _per_source_hits(
            files.get("correctness_goldens.json", [])).items():
        if n == 0:
            raise EvaluationInputError(f"Zero correctness rows for {s} (never a 0.0 gate)")
        values[f"eval.gate.correctness.answer_cited.{s}"] = (hits / n, n)

    # misroute_negation — ONLY category=="misroute" rows of query_processing (LLD 03:113)
    qp_rows = files.get("query_processing_goldens.json", [])
    mis_rows = [r for r in qp_rows if r.get("category") == "misroute"]
    for s in SOURCE_IDS:
        rows = [r for r in mis_rows if s in _split_sources(r)]
        n = len(rows)
        if n == 0:
            raise EvaluationInputError(f"Zero misroute rows for {s} (never a 0.0 gate)")
        # stub predicate: a misroute golden stays a true negative (#24) iff its
        # ideal_answer still routes to its OWN source (goldens encode the route).
        hits = sum(1 for r in rows
                   if _normalize_text(r.get("ideal_answer", ""))
                       .lower().startswith(f"route to {s.lower()}"))
        values[f"eval.gate.routing.misroute_negation.{s}"] = (hits / n, n)

    # snapshot_rowcount — provenance only, per source across ALL files (D15 split)
    for s in SOURCE_IDS:
        n = sum(1 for frows in files.values() for r in frows if s in _split_sources(r))
        values[f"eval.info.snapshot_rowcount.{s}"] = (float(n), n)

    return values

In [12]:
#run_suite Main Function
# Deterministic offline eval — no network, no LLM, no env keys (T-03-2 docstring).
# Output = candidate report envelope {schema_version, generated_utc, metrics}
# (LLD 03:189-191, D40) — corpus_sha256 belongs to the Snapshot, not the report.

import datetime

@dataclass(frozen=True)
class MetricValue:
    metric_id: str
    value: float
    sample_size: int
    computation: Literal["stub"]     # all gate rows are offline-deterministic in Mod 3

def run_suite(goldens_dir: Path = GOLDENS_DIR, docs_dir: Path = DOCS_DIR) -> dict:
    """Pure offline deterministic suite — returns the candidate report envelope."""
    values = evaluate_golden_records(goldens_dir)

    # Mod-3 slice: only CAN-COMPUTE rows (16 gate + 5 snapshot_rowcount = 21).
    # guardrail band8/position_bias and the two latency rows are REGISTERED but
    # computed nightly / Mod-4 — they never enter the offline report (LLD 03:121-124).
    computed = []
    for m in REGISTRY:
        if m.id in values:
            val, n = values[m.id]
            computed.append({
                "metric_id": m.id, "value": val,
                "sample_size": n, "computation": "stub",
            })
    computed.sort(key=lambda mv: mv["metric_id"])   # canonical order (T-03-2)

    return {
        "schema_version": "1.0",
        "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "metrics": computed,
    }

In [13]:
#(Execution Test Drive)

report = run_suite(GOLDENS_DIR, DOCS_DIR)

gate_n = sum(1 for m in report["metrics"] if m["metric_id"].startswith("eval.gate."))
print("run_suite() executed successfully:")
print(f"  - Schema Version : {report['schema_version']}")
print(f"  - Generated UTC  : {report['generated_utc']}")
print(f"  - Total Metrics  : {len(report['metrics'])} (16 gate + 5 snapshot_rowcount)")
print(f"  - Gate rows      : {gate_n}")
print("  - Corpus SHA-256 :", compute_corpus_sha256(DOCS_DIR))

--- /home/dipak/agentic/step9_llmops/eval/goldens/correctness_goldens.json
TOTAL 107 (range 1-1000000000) | FAILURES: NONE
categories: {'conflict': 5, 'multi-source': 1, 'cite': 73, 'degrade': 12, 'abstain': 10, 'misroute': 6}
  edge misroute: 6 (target >=5: OK)
  edge conflict: 5 (target >=5: OK)
  edge abstain: 10 (target >=5: OK)
  edge degrade: 12 (target >=5: OK)
  basic share: 0.0% (cap 30%: OK)
  file-level targets: PASS
--- /home/dipak/agentic/step9_llmops/eval/goldens/query_processing_goldens.json
TOTAL 67 (range 1-1000000000) | FAILURES: NONE
categories: {'conflict': 5, 'multi-source': 5, 'cite': 38, 'misroute': 8, 'basic': 1, 'abstain': 5, 'degrade': 5}
  edge misroute: 8 (target >=5: OK)
  edge conflict: 5 (target >=5: OK)
  edge abstain: 5 (target >=5: OK)
  edge degrade: 5 (target >=5: OK)
  basic share: 1.5% (cap 30%: OK)
  file-level targets: PASS
--- /home/dipak/agentic/step9_llmops/eval/goldens/retriever_goldens.json
TOTAL 139 (range 1-1000000000) | FAILURES: NONE
cat

In [14]:
#Canonical Bytes Determinism Helper
# Deterministic subset = schema_version + metrics sorted by metric_id.
# generated_utc is recording metadata EXCLUDED from byte identity (T-03-2, LLD 03:185-188).

def compute_canonical_bytes(report: dict) -> bytes:
    """Deterministic canonical subset — two identical runs produce identical bytes."""
    canonical_payload = {
        "schema_version": report.get("schema_version", "1.0"),
        "metrics": sorted(report.get("metrics", []), key=lambda m: m["metric_id"]),
    }
    return json.dumps(canonical_payload, indent=2, sort_keys=True).encode("utf-8")


# Assert byte determinism across consecutive runs (T-03-2)
bytes_run1 = compute_canonical_bytes(run_suite(GOLDENS_DIR, DOCS_DIR))
bytes_run2 = compute_canonical_bytes(run_suite(GOLDENS_DIR, DOCS_DIR))
assert bytes_run1 == bytes_run2, "Determinism failure: canonical bytes differ between identical runs!"
print("Determinism check passed: canonical byte outputs are identical across runs.")

--- /home/dipak/agentic/step9_llmops/eval/goldens/correctness_goldens.json
TOTAL 107 (range 1-1000000000) | FAILURES: NONE
categories: {'conflict': 5, 'multi-source': 1, 'cite': 73, 'degrade': 12, 'abstain': 10, 'misroute': 6}
  edge misroute: 6 (target >=5: OK)
  edge conflict: 5 (target >=5: OK)
  edge abstain: 10 (target >=5: OK)
  edge degrade: 12 (target >=5: OK)
  basic share: 0.0% (cap 30%: OK)
  file-level targets: PASS
--- /home/dipak/agentic/step9_llmops/eval/goldens/query_processing_goldens.json
TOTAL 67 (range 1-1000000000) | FAILURES: NONE
categories: {'conflict': 5, 'multi-source': 5, 'cite': 38, 'misroute': 8, 'basic': 1, 'abstain': 5, 'degrade': 5}
  edge misroute: 8 (target >=5: OK)
  edge conflict: 5 (target >=5: OK)
  edge abstain: 5 (target >=5: OK)
  edge degrade: 5 (target >=5: OK)
  basic share: 1.5% (cap 30%: OK)
  file-level targets: PASS
--- /home/dipak/agentic/step9_llmops/eval/goldens/retriever_goldens.json
TOTAL 139 (range 1-1000000000) | FAILURES: NONE
cat

--- /home/dipak/agentic/step9_llmops/eval/goldens/correctness_goldens.json
TOTAL 107 (range 1-1000000000) | FAILURES: NONE
categories: {'conflict': 5, 'multi-source': 1, 'cite': 73, 'degrade': 12, 'abstain': 10, 'misroute': 6}
  edge misroute: 6 (target >=5: OK)
  edge conflict: 5 (target >=5: OK)
  edge abstain: 10 (target >=5: OK)
  edge degrade: 12 (target >=5: OK)
  basic share: 0.0% (cap 30%: OK)
  file-level targets: PASS
--- /home/dipak/agentic/step9_llmops/eval/goldens/query_processing_goldens.json
TOTAL 67 (range 1-1000000000) | FAILURES: NONE
categories: {'conflict': 5, 'multi-source': 5, 'cite': 38, 'misroute': 8, 'basic': 1, 'abstain': 5, 'degrade': 5}
  edge misroute: 8 (target >=5: OK)
  edge conflict: 5 (target >=5: OK)
  edge abstain: 5 (target >=5: OK)
  edge degrade: 5 (target >=5: OK)
  basic share: 1.5% (cap 30%: OK)
  file-level targets: PASS
--- /home/dipak/agentic/step9_llmops/eval/goldens/retriever_goldens.json
TOTAL 139 (range 1-1000000000) | FAILURES: NONE
cat

**Determinism narrative** — canonical-bytes subset (schema_version + sorted metrics, `generated_utc` excluded) → promotes to T-03-2.

**Empty `must_contain` trap** — runtime guard → `EvaluationInputError` (H8); empty goldens dir → exit 3 (T-03-2a).

## Section 3 — Manual exploration / dry runs

Run against the committed goldens, eyeball the report before wiring tests.

In [15]:
# --- Display the candidate report ------------------------------------------
print(f"{'metric_id':<45} {'value':>10} {'n':>5} {'computation'}")
print("-" * 75)
for row in sorted(report["metrics"], key=lambda r: r["metric_id"]):
    print(f"{row['metric_id']:<45} {row['value']:>10.6f} {row['sample_size']:>5} {row['computation']}")

metric_id                                          value     n computation
---------------------------------------------------------------------------
eval.gate.correctness.answer_cited.S1           1.000000    25 stub
eval.gate.correctness.answer_cited.S2           1.000000    22 stub
eval.gate.correctness.answer_cited.S3           1.000000    20 stub
eval.gate.correctness.answer_cited.S4           1.000000    24 stub
eval.gate.correctness.answer_cited.S5           1.000000    21 stub
eval.gate.golden_rules                          1.000000     3 stub
eval.gate.retriever.agreement.S1                1.000000    34 stub
eval.gate.retriever.agreement.S2                1.000000    27 stub
eval.gate.retriever.agreement.S3                1.000000    28 stub
eval.gate.retriever.agreement.S4                1.000000    28 stub
eval.gate.retriever.agreement.S5                1.000000    26 stub
eval.gate.routing.misroute_negation.S1          1.000000     1 stub
eval.gate.routing.misroute_negati

**Expected-value narrative** — every gate row should read 1.0 on committed goldens (T-03-4).

In [16]:
# --- Scratch: deliberately break a golden row (in-memory only) -------------
# Not persisted to disk — just proves the single-flip tolerance floor reacts
# the way Section 1's registry intends (S1 n=34 case, D39).

import copy

_scratch = copy.deepcopy(report["metrics"])
S1_ID_SCRATCH = "eval.gate.retriever.agreement.S1"

for row in _scratch:
    if row["metric_id"] == S1_ID_SCRATCH:
        n = row["sample_size"]
        row["value"] = (n - 1) / n   # simulate one flipped row
        tol = get_metric(S1_ID_SCRATCH).tolerance
        print(f"Simulated one flipped {S1_ID_SCRATCH} row: {n}/{n} -> {n-1}/{n} = {row['value']:.6f}")
        print(f"Registry tolerance floor: {tol:.6f}  (bound = 1.0 - tol = {1.0 - tol:.6f})")
        print("-> below bound, so this WOULD trip the gate once compare() exists (Section 6)")

Simulated one flipped eval.gate.retriever.agreement.S1 row: 34/34 -> 33/34 = 0.970588
Registry tolerance floor: 0.028571  (bound = 1.0 - tol = 0.971429)
-> below bound, so this WOULD trip the gate once compare() exists (Section 6)


Note: this section is throwaway exploration, not part of the promoted module.

## Section 4 — `snapshot` prototyping

Baseline serialization contract — committed, atomic write, provenance fields.

In [17]:
# --- Snapshot dataclass + provenance helpers -----------------------------
import datetime
import subprocess

class ConfigurationError(Exception):
    """Raised for baseline/pointer structural violations -> CLI exit 4 (D36)."""
    pass

def compute_goldens_sha256(goldens_dir: Path = GOLDENS_DIR) -> str:
    """Canonical manifest over eval/goldens/**: sorted relative paths + file bytes."""
    if not goldens_dir.exists() or not goldens_dir.is_dir():
        raise EvaluationInputError(f"Goldens directory not found: {goldens_dir}")
    rel_files = sorted(f.relative_to(goldens_dir) for f in goldens_dir.rglob("*") if f.is_file())
    if not rel_files:
        raise EvaluationInputError(f"No golden files found in: {goldens_dir}")
    hasher = hashlib.sha256()
    for rel in rel_files:
        hasher.update(str(rel).encode("utf-8"))
        hasher.update(b"\x00")
        hasher.update((goldens_dir / rel).read_bytes())
    return hasher.hexdigest()

def get_git_commit(root: Path = ROOT) -> str | None:
    """Cheap, optional provenance — None if not a git repo."""
    try:
        out = subprocess.run(
            ["git", "rev-parse", "HEAD"], cwd=root,
            capture_output=True, text=True, timeout=5, check=True,
        )
        return out.stdout.strip() or None
    except Exception:
        return None

@dataclass
class Snapshot:
    id: str
    created_utc: str
    git_commit: str | None
    schema_version: str
    goldens_sha256: str
    corpus_sha256: str
    metrics: dict[str, dict]   # metric_id -> {"value", "sample_size", "computation"}
    meta: dict

In [18]:
# --- save_snapshot() -------------------------------------------------------
# Atomic write: tmp file + rename. Write-time completeness: all 16 gate rows
# must be present (T-03-5a); guardrail/info rows optional.

import dataclasses

def save_snapshot(snapshot: Snapshot, *, path: Path | None = None) -> None:
    if path is None:
        path = BASELINES_DIR / f"{snapshot.id}.json"

    missing_gates = [gid for gid in gate_metric_ids() if gid not in snapshot.metrics]
    if missing_gates:
        raise EvaluationInputError(
            f"snapshot incomplete — missing gate rows: {missing_gates} (T-03-5a)"
        )

    path.parent.mkdir(parents=True, exist_ok=True)
    payload = dataclasses.asdict(snapshot)

    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    tmp_path.replace(path)   # atomic rename

In [19]:
# --- load_snapshot() + round-trip (T-03-5) --------------------------------
def load_snapshot(path: Path) -> Snapshot:
    """Deserialize a committed baseline; round-trip must preserve every field (LLD 03:213)."""
    if not path.exists() or not path.is_file():
        raise ConfigurationError(f"Baseline file missing: {path} (exit 4, T-03-5b)")
    payload = json.loads(path.read_text(encoding="utf-8"))
    return Snapshot(**payload)

# Round-trip: build a Snapshot from the candidate report, save, load, compare.
# Demo artifacts live in repo .temp scratch (house rule) — real eval/baselines/
# holds ONLY the committed baseline + active pointer (LLD 03:216-218).
demo_dir = ROOT / ".temp" / "nb003" / "scratch"
demo_dir.mkdir(parents=True, exist_ok=True)
snap = Snapshot(
    id="baseline-demo-roundtrip",
    created_utc=report["generated_utc"],
    git_commit=get_git_commit(),
    schema_version=report["schema_version"],
    goldens_sha256=compute_goldens_sha256(GOLDENS_DIR),
    corpus_sha256=compute_corpus_sha256(DOCS_DIR),
    metrics={m["metric_id"]: {"value": m["value"], "sample_size": m["sample_size"], "computation": m["computation"]} for m in report["metrics"]},
    meta={"golden_counts": "313", "source": "NB-003 Section 4 demo"},
)
save_snapshot(snap, path=demo_dir / f"{snap.id}.json")

rt = load_snapshot(demo_dir / "baseline-demo-roundtrip.json")
assert dataclasses.asdict(rt) == dataclasses.asdict(snap), "T-03-5 round-trip mismatch"
print(f"T-03-5 round-trip OK: {len(rt.metrics)} metrics, git_commit={rt.git_commit!r}")
print(f"  (demo snapshot saved to repo {demo_dir} — NOT eval/baselines/)")


T-03-5 round-trip OK: 21 metrics, git_commit='a8f50c5cb30faba7edeb46429e654d2c3ffabd27'
  (demo snapshot saved to repo /home/dipak/agentic/step9_llmops/.temp/nb003/scratch — NOT eval/baselines/)


In [20]:
# --- Write-time completeness: all 16 gate rows present (T-03-5a) -----------
rt_gates = [mid for mid in rt.metrics if mid.startswith("eval.gate.")]
expected = set(gate_metric_ids())
missing = expected - set(rt.metrics)
assert not missing, f"T-03-5a missing gate rows: {sorted(missing)}"
assert len(rt_gates) == 16, f"T-03-5a expected 16 gate rows, got {len(rt_gates)}"
assert rt.schema_version == "1.0"
assert rt.goldens_sha256 and rt.corpus_sha256
print(f"T-03-5a OK: {len(rt_gates)} gate rows present, goldens={rt.goldens_sha256[:12]}... corpus={rt.corpus_sha256[:12]}...")


T-03-5a OK: 16 gate rows present, goldens=0a93278588bc... corpus=0ed886f88b3b...


**Canonical pointer (`active.json`)** — schema `{schema_version, baseline_id, path}`, path-safety rules (basename only, no traversal), resolved before compare (T-03-5b).

In [21]:
# --- Canonical pointer active.json (T-03-5b) -------------------------------
# Real pointer lives in eval/baselines; DEMO pointer + battery live in repo
# .temp scratch so every demo/battery case stays OUT of the committed dir.
ACTIVE_POINTER = BASELINES_DIR / "active.json"
demo_dir = ROOT / ".temp" / "nb003" / "scratch"

def write_active_pointer(snapshot: Snapshot, *, pointer: Path = ACTIVE_POINTER) -> None:
    """Atomic pointer rewrite {schema_version, baseline_id, path}; path = basename only (LLD 03:220-226)."""
    payload = json.dumps({
        "schema_version": snapshot.schema_version,
        "baseline_id": snapshot.id,
        "path": f"{snapshot.id}.json",
    }, indent=2, sort_keys=True) + "\n"
    tmp = pointer.with_suffix(pointer.suffix + ".tmp")
    tmp.write_text(payload, encoding="utf-8")
    tmp.replace(pointer)   # atomic rename (H12)

def resolve_active_baseline(pointer: Path = ACTIVE_POINTER,
                            *, base_dir: Path = BASELINES_DIR) -> Path:
    """Resolve the active pointer BEFORE compare; any violation -> ConfigurationError -> exit 4.
    base_dir defaults to eval/baselines; demo pointers resolve against .temp scratch."""
    if not pointer.exists():
        raise ConfigurationError(f"active.json missing: {pointer} (exit 4, T-03-5b)")
    try:
        pj = json.loads(pointer.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ConfigurationError(f"malformed active.json: {e} (exit 4, T-03-5b)") from e
    for field in ("schema_version", "baseline_id", "path"):
        if not isinstance(pj.get(field), str) or not pj[field]:
            raise ConfigurationError(f"active.json field missing/empty: {field!r} (exit 4, T-03-5b)")

    name = Path(pj["path"])
    if name.name != str(name) or ".." in name.parts or name.is_absolute() or "\\" in str(pj["path"]):
        raise ConfigurationError(f"pointer path is not a safe basename: {pj['path']!r} (exit 4, T-03-5b)")
    if name.suffix != ".json":
        raise ConfigurationError(f"pointer path must end .json: {pj['path']!r} (exit 4, T-03-5b)")

    base_abs = Path(base_dir).resolve()
    baseline_path = (base_abs / name).resolve()
    if not baseline_path.is_relative_to(base_abs):
        raise ConfigurationError(f"pointer path escapes baseline dir: {pj['path']!r} (exit 4, T-03-5b)")
    if not baseline_path.is_file():
        raise ConfigurationError(f"pointer target missing: {baseline_path} (exit 4, T-03-5b)")

    loaded = load_snapshot(baseline_path)
    if loaded.id != pj["baseline_id"]:
        raise ConfigurationError(f"baseline_id mismatch: pointer={pj['baseline_id']!r} snapshot={loaded.id!r} (exit 4, T-03-5b)")
    return baseline_path

# Happy path: point a DEMO pointer at the round-trip baseline, then resolve it.
demo_pointer = demo_dir / "active.json"
write_active_pointer(rt, pointer=demo_pointer)
resolved = resolve_active_baseline(demo_pointer, base_dir=demo_dir)
assert resolved == (demo_dir / f"{rt.id}.json").resolve()
print(f"T-03-5b happy path OK: demo active.json -> {resolved.name} (in .temp scratch)")

# Malformed-pointer battery (T-03-5b): every case must raise ConfigurationError.
pointer_bad = demo_dir / "active_battery.json"

def expect_config_error(label: str, fn):
    try:
        fn()
    except ConfigurationError as e:
        print(f"  [battery::{label}] OK -> {str(e)[:60]}...")
        return
    raise AssertionError(f"battery::{label} did NOT raise ConfigurationError")

# 1. missing pointer
expect_config_error("missing-active", lambda: resolve_active_baseline(demo_dir / "does_not_exist.json", base_dir=demo_dir))
# 2. malformed JSON
pointer_bad.write_text("{ not json", encoding="utf-8")
expect_config_error("malformed-json", lambda: resolve_active_baseline(pointer_bad, base_dir=demo_dir))
# 3. traversal
pointer_bad.write_text('{"schema_version":"1.0","baseline_id":"x","path":"../other.json"}', encoding="utf-8")
expect_config_error("traversal", lambda: resolve_active_baseline(pointer_bad, base_dir=demo_dir))
# 4. absolute path
pointer_bad.write_text('{"schema_version":"1.0","baseline_id":"x","path":"/tmp/escape.json"}', encoding="utf-8")
expect_config_error("absolute", lambda: resolve_active_baseline(pointer_bad, base_dir=demo_dir))
# 5. backslash
pointer_bad.write_text('{"schema_version":"1.0","baseline_id":"x","path":"..\\\\escape.json"}', encoding="utf-8")
expect_config_error("backslash", lambda: resolve_active_baseline(pointer_bad, base_dir=demo_dir))
# 6. wrong extension
pointer_bad.write_text('{"schema_version":"1.0","baseline_id":"x","path":"baseline.txt"}', encoding="utf-8")
expect_config_error("wrong-extension", lambda: resolve_active_baseline(pointer_bad, base_dir=demo_dir))
# 7. target missing (valid name, no file)
pointer_bad.write_text('{"schema_version":"1.0","baseline_id":"x","path":"missing.json"}', encoding="utf-8")
expect_config_error("target-missing", lambda: resolve_active_baseline(pointer_bad, base_dir=demo_dir))
# 8. baseline_id mismatch
pointer_bad.write_text('{"schema_version":"1.0","baseline_id":"wrong-id","path":"' + Path(rt.id).name + '.json"}', encoding="utf-8")
expect_config_error("id-mismatch", lambda: resolve_active_baseline(pointer_bad, base_dir=demo_dir))
print("T-03-5b battery: all malformed-pointer cases raised ConfigurationError (in .temp scratch)")

# IMPORTANT: demo pointers/battery never touch eval/baselines. The REAL active.json
# is written by Section 5 (cell 41/42) pointing at the committed baseline.


T-03-5b happy path OK: demo active.json -> baseline-demo-roundtrip.json (in .temp scratch)
  [battery::missing-active] OK -> active.json missing: /home/dipak/agentic/step9_llmops/.temp/...
  [battery::malformed-json] OK -> malformed active.json: Expecting property name enclosed in d...
  [battery::traversal] OK -> pointer path is not a safe basename: '../other.json' (exit 4...
  [battery::absolute] OK -> pointer path is not a safe basename: '/tmp/escape.json' (exi...
  [battery::backslash] OK -> pointer path is not a safe basename: '..\\escape.json' (exit...
  [battery::wrong-extension] OK -> pointer path must end .json: 'baseline.txt' (exit 4, T-03-5b...
  [battery::target-missing] OK -> pointer target missing: /home/dipak/agentic/step9_llmops/.te...
  [battery::id-mismatch] OK -> baseline_id mismatch: pointer='wrong-id' snapshot='baseline-...
T-03-5b battery: all malformed-pointer cases raised ConfigurationError (in .temp scratch)


**Baseline-change policy** — baseline + pointer change ship in the same commit (registered change, D19-analogous).

## Section 5 — Build + commit the first baseline

One-time act: create `eval/baselines/<id>.json` + `active.json`.

In [22]:
# --- Build the first baseline: eval/baselines/<id>.json (T-03-5 + task 03:22) ---
# Reproducible recipe for the committed baseline. id chosen by the learner (LLD 03:203).
BASELINE_ID = "baseline-2026-09-15"

def compute_golden_counts(goldens_dir: Path = GOLDENS_DIR) -> dict:
    """Deterministic per-source counts (retriever / correctness / misroute) for meta."""
    counts = {"retriever": {}, "correctness": {}, "misroute": {}}
    def key_of(s, tag):
        for k in counts:
            counts[k][s] = counts[k].get(s, 0) + 1
    for fp_name, tag in (("retriever_goldens.json", "retriever"),
                         ("correctness_goldens.json", "correctness")):
        rows = json.loads((goldens_dir / fp_name).read_text(encoding="utf-8"))
        for r in rows:
            for s in _split_sources(r):
                counts[tag][s] = counts[tag].get(s, 0) + 1
    qp = json.loads((goldens_dir / "query_processing_goldens.json").read_text(encoding="utf-8"))
    for r in qp:
        if r.get("category") == "misroute":
            for s in _split_sources(r):
                counts["misroute"][s] = counts["misroute"].get(s, 0) + 1
    return counts

baseline = Snapshot(
    id=BASELINE_ID,
    created_utc=report["generated_utc"],
    git_commit=get_git_commit(),
    schema_version=report["schema_version"],
    goldens_sha256=compute_goldens_sha256(GOLDENS_DIR),
    corpus_sha256=compute_corpus_sha256(DOCS_DIR),
    metrics={m["metric_id"]: {"value": m["value"], "sample_size": m["sample_size"], "computation": m["computation"]}
             for m in report["metrics"]},
    meta={"golden_counts": compute_golden_counts(), "notes": "Mod 3 initial baseline"},
)
save_snapshot(baseline)

baseline_path = BASELINES_DIR / f"{BASELINE_ID}.json"
assert baseline_path.is_file(), "baseline not written"
loaded = load_snapshot(baseline_path)
missing = set(gate_metric_ids()) - set(loaded.metrics)
assert not missing, f"baseline incomplete: {sorted(missing)}"
print(f"Baseline written: {baseline_path}")
print(f"  id={loaded.id} schema={loaded.schema_version} git={loaded.git_commit}")
print(f"  metrics={len(loaded.metrics)} (16 gate + 5 rowcount) gates_ok={len([m for m in loaded.metrics if m.startswith('eval.gate.')])}")


Baseline written: /home/dipak/agentic/step9_llmops/eval/baselines/baseline-2026-09-15.json
  id=baseline-2026-09-15 schema=1.0 git=a8f50c5cb30faba7edeb46429e654d2c3ffabd27
  metrics=21 (16 gate + 5 rowcount) gates_ok=16


In [23]:
# --- Point the canonical pointer at the baseline (T-03-5b) -----------------
# Baseline + pointer change ship in the same commit (registered change, task 03, D19-analogous).

write_active_pointer(baseline)
resolved = resolve_active_baseline()
assert resolved == baseline_path.resolve(), "active.json does not resolve to the written baseline"
print(f"active.json -> {resolved.name}  (baseline + pointer ready for one commit)")
print("Pointer content:", (ACTIVE_POINTER).read_text(encoding="utf-8").strip().replace(chr(10), " "))


active.json -> baseline-2026-09-15.json  (baseline + pointer ready for one commit)
Pointer content: {   "baseline_id": "baseline-2026-09-15",   "path": "baseline-2026-09-15.json",   "schema_version": "1.0" }


Reminder: baseline commit is reviewed, not silent (D19-analogous registered change).

## Section 6 — `compare` prototyping

Verdict engine — 11-step precedence, structural checks BEFORE value comparison.

In [24]:
# --- Verdict + tolerance math (LLD 03:231-235, 265-269) --------------------
# compare() never rounds metric values before verdicting (INVARIANT, D41);
# boundary comparisons use raw float math against the computed bound.

@dataclass(frozen=True)
class Verdict:
    code: int                # 0=PASS 1=FAIL 2=REVIEW   (3/4 are error classes, raised)
    regressed: tuple[str, ...]
    detail: dict[str, str]

def _worse_than_baseline(m: Metric, base: float, cand: float) -> bool:
    """True when cand moved beyond the computed tolerance bound (inclusive band)."""
    if m.tolerance_unit == "absolute":
        bound = m.tolerance
        return (cand < base - bound) if m.direction == "higher" else (cand > base + bound)
    # relative
    if m.direction == "higher":
        return cand < base * (1.0 - m.tolerance)
    return cand > base * (1.0 + m.tolerance)

print("Verdict + tolerance math defined (no rounding — INVARIANT).")


Verdict + tolerance math defined (no rounding — INVARIANT).


In [25]:
# --- compare(): 11-step registry-driven precedence (LLD 03:237-275) --------
# Structural checks BEFORE value comparison. compare() returns ONLY a Verdict;
# error classes are RAISED and mapped to exit 3/4 at the CLI boundary (D36).

def get_metric_or_none(metric_id: str) -> Metric | None:
    try:
        return get_metric(metric_id)
    except KeyError:
        return None

def compare(baseline: Snapshot, candidate: list[MetricValue]) -> Verdict:
    cand = {mv.metric_id: mv for mv in candidate}
    regressed: list[str] = []
    review: list[str] = []
    detail: dict[str, str] = {}

    # Step 2 (candidate structure): duplicate candidate ids -> EvaluationInputError (exit 3)
    from collections import Counter
    dupes = [mid for mid, n in Counter(mv.metric_id for mv in candidate).items() if n > 1]
    if dupes:
        raise EvaluationInputError(f"duplicate candidate metric ids: {sorted(dupes)} (exit 3)")

    # Step 4 (unknown/unregistered candidate id -> FAIL, T-03-8g) — before any value math.
    for mid in cand:
        m = get_metric_or_none(mid)
        if m is None:
            regressed.append(mid)
            detail[mid] = f"unregistered candidate metric id -> FAIL (T-03-8g)"
            continue

    # Step 5 (baseline structure): zero baseline under relative tolerance -> ConfigError (4).
    # Applies to ANY compared row (gate or guardrail — latency.p95 is the relative row,
    # D10/D36); info rows are recorded, never compared, so they are exempt (T-03-8e).
    # Also missing baseline gate rows that the candidate HAS -> ConfigError (4) (T-03-8d).
    for mid, cv in cand.items():
        m = get_metric_or_none(mid)
        if m is None or m.kind == "info":
            continue
        b_raw = baseline.metrics.get(mid)
        if b_raw is not None and m.tolerance_unit == "relative" and b_raw["value"] == 0.0:
            raise ConfigurationError(f"relative tolerance undefined at zero baseline: {mid} (exit 4, T-03-8e)")

    for mid in gate_metric_ids():
        m = get_metric(mid)
        b_raw = baseline.metrics.get(mid)
        c_raw = cand.get(mid)

        # Step 4 first: gate missing in candidate -> FAIL (silent-drop protection, T-03-8b),
        # INCLUDING when the same gate is missing from BOTH (T-03-8h — candidate-missing fires before baseline-missing).
        if c_raw is None:
            regressed.append(mid)
            detail[mid] = f"gate missing in candidate (T-03-8b/8h)"
            continue
        # Step 5: gate missing in baseline while candidate HAS it -> ConfigError (4, T-03-8d)
        if b_raw is None:
            raise ConfigurationError(f"baseline missing gate row that candidate has: {mid} (exit 4, T-03-8d)")

        # Step 3: coverage regression — candidate sample_size != registered expected_sample_size (D39, T-03-8i)
        if m.expected_sample_size is not None and c_raw.sample_size != m.expected_sample_size:
            regressed.append(mid)
            detail[mid] = (f"coverage regression n={c_raw.sample_size} != registered "
                           f"{m.expected_sample_size} (T-03-8i)")
            continue

        # Steps 6/7: gate value comparison (inclusive computed bound, never rounded)
        if _worse_than_baseline(m, b_raw["value"], c_raw.value):
            regressed.append(mid)
            detail[mid] = f"gate value {c_raw.value} vs baseline {b_raw['value']} tol {m.tolerance:.6g} (T-03-7)"

    # Steps 8/9: guardrails — REVIEW only when present in BOTH; else skip w/ provenance (H2, T-03-8c)
    for m in REGISTRY:
        if m.kind != "guardrail":
            continue
        b_raw = baseline.metrics.get(m.id)
        c_raw = cand.get(m.id)
        if b_raw is None or c_raw is None:
            if b_raw is not None:
                detail[m.id] = f"guardrail missing in candidate — skipped, not verdict (T-03-8c)"
            continue
        if _worse_than_baseline(m, b_raw["value"], c_raw.value):
            review.append(m.id)
            detail[m.id] = f"guardrail regressed: {c_raw.value} vs baseline {b_raw['value']} tol {m.tolerance:.6g} (T-03-8)"

    # Step 10: info rows recorded, never a verdict — nothing to do.

    # Step 11: precedence FAIL > REVIEW > PASS — a gate failure is NEVER downgraded (T-03-8a).
    if regressed:
        return Verdict(1, tuple(sorted(set(regressed))), detail)
    if review:
        return Verdict(2, tuple(sorted(set(review))), detail)
    return Verdict(0, (), detail)

print("compare() defined — 11-step precedence, structural checks first.")


compare() defined — 11-step precedence, structural checks first.


In [26]:
# --- Demo scaffolding: build candidate MetricValue lists from the real report ---
def _mv(metric_id: str, value: float, sample_size: int | None = None) -> MetricValue:
    m = get_metric_or_none(metric_id)
    n = sample_size if sample_size is not None else ((m.expected_sample_size if m else None) or 0)
    return MetricValue(metric_id=metric_id, value=value, sample_size=n, computation="stub")

def candidate_like_baseline(overrides: dict | None = None) -> list[MetricValue]:
    """Start from the committed baseline's gate values; apply value overrides."""
    overrides = overrides or {}
    out = []
    for mid in gate_metric_ids():
        m = get_metric(mid)
        base_val = baseline.metrics[mid]["value"]
        val, n = overrides.get(mid, (base_val, m.expected_sample_size))
        out.append(_mv(mid, val, n))
    return out

print("demo scaffolding ready (helper functions only).")


demo scaffolding ready (helper functions only).


In [27]:
# --- T-03-6 PASS + T-03-6a boundary PASS (inclusive computed bound) -------
v = compare(baseline, candidate_like_baseline())
assert v.code == 0 and v.regressed == (), f"T-03-6 expected PASS, got {v}"
print(f"T-03-6 PASS ok: exit {v.code}, regressed={v.regressed}")

# Boundary PASS: candidate exactly at baseline - tol (inclusive) -> still PASS.
# retriever.agreement.S1 tol = 1/35 = 0.0285714; bound = 34/35.
from fractions import Fraction
bound = Fraction(1, 1) - Fraction(1, 35)
v_b = compare(baseline, candidate_like_baseline({
    "eval.gate.retriever.agreement.S1": (float(bound), 34),
}))
assert v_b.code == 0, f"T-03-6a expected PASS at the boundary, got {v_b.code} (detail={v_b.detail})"
print(f"T-03-6a boundary PASS ok: value at 1-1/35 = {float(bound):.6f} still PASS (inclusive)")


T-03-6 PASS ok: exit 0, regressed=()
T-03-6a boundary PASS ok: value at 1-1/35 = 0.971429 still PASS (inclusive)


In [28]:
# --- T-03-7 FAIL: one gate drifts below its floor --------------------------
v = compare(baseline, candidate_like_baseline({
    "eval.gate.retriever.agreement.S1": (33 / 34, 34),
}))
assert v.code == 1, f"T-03-7 expected FAIL, got {v.code}"
assert "eval.gate.retriever.agreement.S1" in v.regressed
print(f"T-03-7 FAIL ok: code={v.code}, regressed={v.regressed}")
print(f"  detail: {v.detail['eval.gate.retriever.agreement.S1']}")


T-03-7 FAIL ok: code=1, regressed=('eval.gate.retriever.agreement.S1',)
  detail: gate value 0.9705882352941176 vs baseline 1.0 tol 0.0285714 (T-03-7)


In [29]:
# --- T-03-6b single-flip floor boundary: (n-1)/n must strictly trip ---------
# D39: a one-row degradation (n-1)/n sits strictly below n/(n+1). Committed
# denominator n is the source of truth (D40) — never derived from candidate.
n = get_metric("eval.gate.retriever.agreement.S1").expected_sample_size  # 34
v = compare(baseline, candidate_like_baseline({
    "eval.gate.retriever.agreement.S1": ((n - 1) / n, n),
}))
assert v.code == 1, f"T-03-6b single-flip expected FAIL, got {v.code}"
assert "eval.gate.retriever.agreement.S1" in v.regressed
print(f"T-03-6b single-flip floor ok: 33/34={33/34:.6f} < 34/35={34/35:.6f} -> FAIL(1)")

v_full = compare(baseline, candidate_like_baseline())
assert v_full.code == 0
print("  full-set n/n regression still PASS(0)")


T-03-6b single-flip floor ok: 33/34=0.970588 < 34/35=0.971429 -> FAIL(1)
  full-set n/n regression still PASS(0)


In [30]:
# --- T-03-8a gate beats guardrail priority ---------------------------------
cand = candidate_like_baseline({"eval.gate.retriever.agreement.S1": (0.90, 34)})
cand.append(_mv("eval.guardrail.position_bias.agreement", 0.80, 20))
v = compare(baseline, cand)
assert v.code == 1, f"T-03-8a expected FAIL(1) — gate never downgraded by guardrail, got {v.code}"
assert "eval.gate.retriever.agreement.S1" in v.regressed
print(f"T-03-8a gate-beats-guardrail ok: code={v.code} (gate FAIL wins over guardrail REVIEW)")


T-03-8a gate-beats-guardrail ok: code=1 (gate FAIL wins over guardrail REVIEW)


In [31]:
# --- T-03-8b missing candidate gate -> FAIL; T-03-8g unregistered id -> FAIL ---
cand_no_gate = [mv for mv in candidate_like_baseline()
                if mv.metric_id != "eval.gate.correctness.answer_cited.S3"]
v = compare(baseline, cand_no_gate)
assert v.code == 1 and "eval.gate.correctness.answer_cited.S3" in v.regressed
print(f"T-03-8b silent-drop ok: code={v.code}, regressed={v.regressed}")

cand_unknown = candidate_like_baseline() + [_mv("eval.gate.retriever.agreement.X1", 1.0, 10)]
v = compare(baseline, cand_unknown)
assert v.code == 1 and "eval.gate.retriever.agreement.X1" in v.regressed
print(f"T-03-8g unregistered id ok: code={v.code}, regressed={v.regressed}")
print("Section 6 compare() demos all green.")


T-03-8b silent-drop ok: code=1, regressed=('eval.gate.correctness.answer_cited.S3',)
T-03-8g unregistered id ok: code=1, regressed=('eval.gate.retriever.agreement.X1',)
Section 6 compare() demos all green.


**Boundary semantics** — inclusive band, computed bound (e.g. `n/(n+1)` floor), no rounding before verdicting (compare INVARIANT).

## Section 7 — CLI (`main`) prototyping

`--baseline` vs `--active`, `--candidate`, exit code 0–4 contract.

In [32]:
# --- CLI main() — exit code IS the verdict (T-03-9/T-03-9b, D36) ------------
# argparse usage errors MUST exit 3, never argparse's native exit 2 (2 == REVIEW).

class _UsageError(Exception):
    """argparse usage error -> mapped to exit 3 (D36/D41)."""

import argparse as _argparse

class _GateParser(_argparse.ArgumentParser):
    def error(self, message: str):
        raise _UsageError(f"usage error: {message}")

def _parse_report(path: str) -> list[MetricValue]:
    """Deserialize the candidate envelope {schema_version, generated_utc, metrics} into MetricValue list."""
    from pathlib import Path as _P
    p = _P(path)
    if not p.exists():
        raise EvaluationInputError(f"candidate report not found: {path} (exit 3)")
    try:
        payload = json.loads(p.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise EvaluationInputError(f"malformed candidate report JSON: {e} (exit 3)") from e
    metrics = payload.get("metrics")
    if not isinstance(metrics, list):
        raise EvaluationInputError("candidate report has no 'metrics' list (exit 3)")
    out: list[MetricValue] = []
    for m in metrics:
        if not isinstance(m, dict) or not {"metric_id", "value", "sample_size", "computation"} <= set(m):
            raise EvaluationInputError(f"malformed metric row in candidate report (exit 3): {m!r:.80}")
        out.append(MetricValue(metric_id=m["metric_id"], value=float(m["value"]),
                               sample_size=int(m["sample_size"]), computation=m["computation"]))
    return out

def _load_baseline(path: str | None, use_active: bool) -> Snapshot:
    if use_active:
        resolved = resolve_active_baseline()
        return load_snapshot(resolved)
    if path is None:
        raise ConfigurationError("neither --baseline nor --active given (exit 4)")
    p = Path(path)
    if not p.is_file():
        raise ConfigurationError(f"baseline file missing: {p} (exit 4)")
    return load_snapshot(p)

def main(argv: list[str] | None = None) -> int:
    parser = _GateParser(prog="llmops.eval.compare",
                         description="Offline regression gate — exit 0=PASS 1=FAIL 2=REVIEW 3=input 4=config")
    g = parser.add_mutually_exclusive_group(required=True)
    g.add_argument("--baseline", metavar="PATH", help="committed baseline snapshot JSON")
    g.add_argument("--active", action="store_true", help="resolve eval/baselines/active.json instead")
    parser.add_argument("--candidate", metavar="PATH", required=True, help="candidate report JSON (run_suite output)")
    try:
        args = parser.parse_args(argv)
        baseline = _load_baseline(args.baseline, args.active)
        candidate = _parse_report(args.candidate)
        verdict = compare(baseline, candidate)
        print(f"[gates] verdict={verdict.code} regressed={list(verdict.regressed) or 'none'}")
        for k, v in verdict.detail.items():
            print(f"        detail {k}: {v}")
        return verdict.code
    except _UsageError as e:
        print(f"[gates] usage error -> exit 3: {e}")
        return 3
    except EvaluationInputError as e:
        print(f"[gates] input error -> exit 3: {e}")
        return 3
    except ConfigurationError as e:
        print(f"[gates] config error -> exit 4: {e}")
        return 4

print("main() defined — exit codes 0..4, argparse usage errors -> 3.")


main() defined — exit codes 0..4, argparse usage errors -> 3.


In [33]:
# --- CLI happy-path demo (T-03-9): PASS against the committed baseline ----
# Scratch stays in-repo under .temp/nb003/ (house rule) — NOT /tmp.
scratch = ROOT / ".temp" / "nb003" / "scratch"
scratch.mkdir(parents=True, exist_ok=True)
cand_path = scratch / "candidate_report.json"
cand_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")

rc = main(["--baseline", str(BASELINES_DIR / f"{BASELINE_ID}.json"), "--candidate", str(cand_path)])
assert rc == 0, f"expected PASS(0), got {rc}"

rc2 = main(["--active", "--candidate", str(cand_path)])
assert rc2 == 0, f"--active path expected PASS(0), got {rc2}"
print("CLI happy path: --baseline and --active both PASS(0). Exit code IS the verdict.")
print(f"scratch candidate: {cand_path}")


[gates] verdict=0 regressed=none
[gates] verdict=0 regressed=none
CLI happy path: --baseline and --active both PASS(0). Exit code IS the verdict.
scratch candidate: /home/dipak/agentic/step9_llmops/.temp/nb003/scratch/candidate_report.json


**Error taxonomy recap (0/1/2/3/4)** — 0=PASS, 1=FAIL (regression incl. unknown candidate id, missing candidate gate, coverage regression), 2=REVIEW, 3=EvaluationInputError (malformed report / duplicate ids / empty goldens), 4=ConfigurationError (baseline missing / pointer violations / missing baseline gate / zero-baseline relative). argparse usage errors → 3 via custom `error()` override (native exit 2 reserved for REVIEW, D41).

## Section 8 — Interactive verification against the test matrix

Representative cases from the LLD test matrix as notebook demos (NOT a replacement for `tests/test_gates.py`).

In [34]:
# Expected exit: 0 (T-03-6 — candidate == baseline PASS)
def _write_candidate(metrics: list[MetricValue], name: str) -> Path:
    p = scratch / name
    payload = {"schema_version": "1.0", "generated_utc": report["generated_utc"],
               "metrics": [mv.__dict__ for mv in metrics]}
    p.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return p

rc = main(["--baseline", str(BASELINES_DIR / f"{BASELINE_ID}.json"),
           "--candidate", str(_write_candidate(candidate_like_baseline(), "cand_06_pass.json"))])
print(f"T-03-6: rc={rc} (expect 0)")
assert rc == 0


[gates] verdict=0 regressed=none
T-03-6: rc=0 (expect 0)


In [35]:
# Expected exit: 0 (T-03-6a — boundary inclusive PASS)
from fractions import Fraction
bound = float(Fraction(34, 35))
cand = candidate_like_baseline({"eval.gate.retriever.agreement.S1": (bound, 34)})
rc = main(["--baseline", str(BASELINES_DIR / f"{BASELINE_ID}.json"),
           "--candidate", str(_write_candidate(cand, "cand_06a_boundary.json"))])
print(f"T-03-6a: rc={rc} (expect 0 — inclusive computed bound {bound:.6f})")
assert rc == 0


[gates] verdict=0 regressed=none
T-03-6a: rc=0 (expect 0 — inclusive computed bound 0.971429)


In [36]:
# Expected exit: 1 (T-03-6b — single-flip floor: 33/34 strictly below 34/35)
n = get_metric("eval.gate.retriever.agreement.S1").expected_sample_size
cand = candidate_like_baseline({"eval.gate.retriever.agreement.S1": ((n - 1) / n, n)})
rc = main(["--baseline", str(BASELINES_DIR / f"{BASELINE_ID}.json"),
           "--candidate", str(_write_candidate(cand, "cand_06b_flip.json"))])
print(f"T-03-6b: rc={rc} (expect 1 — single golden flip trips the gate)")
assert rc == 1


[gates] verdict=1 regressed=['eval.gate.retriever.agreement.S1']
        detail eval.gate.retriever.agreement.S1: gate value 0.9705882352941176 vs baseline 1.0 tol 0.0285714 (T-03-7)
T-03-6b: rc=1 (expect 1 — single golden flip trips the gate)


In [37]:
# Expected exit: 1 (T-03-7 — gate value below floor -> FAIL)
cand = candidate_like_baseline({"eval.gate.retriever.agreement.S1": (0.90, 34)})
rc = main(["--baseline", str(BASELINES_DIR / f"{BASELINE_ID}.json"),
           "--candidate", str(_write_candidate(cand, "cand_07_fail.json"))])
print(f"T-03-7: rc={rc} (expect 1 — regressed gate named in detail)")
assert rc == 1


[gates] verdict=1 regressed=['eval.gate.retriever.agreement.S1']
        detail eval.gate.retriever.agreement.S1: gate value 0.9 vs baseline 1.0 tol 0.0285714 (T-03-7)
T-03-7: rc=1 (expect 1 — regressed gate named in detail)


In [38]:
# Expected exit: 1 (T-03-8a — gate FAIL beats guardrail REVIEW; never a REVIEW pass)
cand = candidate_like_baseline({"eval.gate.retriever.agreement.S1": (0.90, 34)})
cand.append(_mv("eval.guardrail.position_bias.agreement", 0.80, 20))
rc = main(["--baseline", str(BASELINES_DIR / f"{BASELINE_ID}.json"),
           "--candidate", str(_write_candidate(cand, "cand_08a_gate_priority.json"))])
print(f"T-03-8a: rc={rc} (expect 1 — gate FAIL wins over guardrail drift)")
assert rc == 1


[gates] verdict=1 regressed=['eval.gate.retriever.agreement.S1']
        detail eval.gate.retriever.agreement.S1: gate value 0.9 vs baseline 1.0 tol 0.0285714 (T-03-7)
T-03-8a: rc=1 (expect 1 — gate FAIL wins over guardrail drift)


In [39]:
# Expected exit: 1 (T-03-8b — missing candidate gate = silent-drop protection)
cand = [mv for mv in candidate_like_baseline()
        if mv.metric_id != "eval.gate.correctness.answer_cited.S3"]
rc = main(["--baseline", str(BASELINES_DIR / f"{BASELINE_ID}.json"),
           "--candidate", str(_write_candidate(cand, "cand_08b_missing_gate.json"))])
print(f"T-03-8b: rc={rc} (expect 1 — suite that drops a gate is a FAIL)")
assert rc == 1


[gates] verdict=1 regressed=['eval.gate.correctness.answer_cited.S3']
        detail eval.gate.correctness.answer_cited.S3: gate missing in candidate (T-03-8b/8h)
T-03-8b: rc=1 (expect 1 — suite that drops a gate is a FAIL)


In [40]:
# Expected exit: 0 (T-03-8c — guardrail present only in candidate is SKIPPED, never a verdict)
# Committed baseline has no guardrail rows (Mod 3 is gate-only). A candidate that adds a
# REGISTERED guardrail row is legal: guardrail logic REQUIRES presence in BOTH, so it must
# be skipped with a provenance note — never a verdict. (H2, T-03-8c)
cand = candidate_like_baseline()
cand.append(_mv("eval.guardrail.position_bias.agreement", 0.80, 20))
rc = main(["--baseline", str(BASELINES_DIR / f"{BASELINE_ID}.json"),
           "--candidate", str(_write_candidate(cand, "cand_08c_guardrail_skip.json"))])
print(f"T-03-8c: rc={rc} (expect 0 — guardrail not in baseline -> skipped)")
assert rc == 0


[gates] verdict=0 regressed=none
T-03-8c: rc=0 (expect 0 — guardrail not in baseline -> skipped)


In [41]:
# Expected exit: 1 (T-03-8h — same gate missing from BOTH baseline & candidate -> FAIL,
# candidate-missing fires before baseline-missing: silent-drop protection wins)
# NOTE: save_snapshot refuses incomplete baselines (T-03-5a) — deliberately. To exercise
# the both-missing comparison we hand-write a legacy-style JSON that predates the
# completeness gate (exactly the artifact a both-missing scenario implies).
cand_8h = [mv for mv in candidate_like_baseline()
           if mv.metric_id != "eval.gate.retriever.agreement.S2"]
b8h_metrics = {k: v for k, v in baseline.metrics.items()
               if k != "eval.gate.retriever.agreement.S2"}
(scratch / "baseline_8h.json").write_text(json.dumps({
    "id": "baseline-8h", "created_utc": baseline.created_utc, "git_commit": baseline.git_commit,
    "schema_version": "1.0", "goldens_sha256": baseline.goldens_sha256,
    "corpus_sha256": baseline.corpus_sha256, "metrics": b8h_metrics, "meta": baseline.meta,
}, indent=2, sort_keys=True) + "\n", encoding="utf-8")
rc = main(["--baseline", str(scratch / "baseline_8h.json"),
           "--candidate", str(_write_candidate(cand_8h, "cand_08h_both_missing.json"))])
print(f"T-03-8h: rc={rc} (expect 1 — S2 missing from candidate fires first, not ConfigError(4))")
assert rc == 1


[gates] verdict=1 regressed=['eval.gate.retriever.agreement.S2']
        detail eval.gate.retriever.agreement.S2: gate missing in candidate (T-03-8b/8h)
T-03-8h: rc=1 (expect 1 — S2 missing from candidate fires first, not ConfigError(4))


In [42]:
# Expected exit: 4 (T-03-8d — baseline missing a gate row the candidate HAS)
# Same provenance note as T-03-8h: save_snapshot REFUSES incomplete baselines (T-03-5a),
# so a missing-row baseline can only exist as a legacy/manual artifact — hand-written below.
b8d_metrics = {k: v for k, v in baseline.metrics.items()
               if k != "eval.gate.routing.misroute_negation.S1"}
(scratch / "baseline_8d.json").write_text(json.dumps({
    "id": "baseline-8d", "created_utc": baseline.created_utc, "git_commit": baseline.git_commit,
    "schema_version": "1.0", "goldens_sha256": baseline.goldens_sha256,
    "corpus_sha256": baseline.corpus_sha256, "metrics": b8d_metrics, "meta": baseline.meta,
}, indent=2, sort_keys=True) + "\n", encoding="utf-8")
rc = main(["--baseline", str(scratch / "baseline_8d.json"),
           "--candidate", str(_write_candidate(candidate_like_baseline(), "cand_08d.json"))])
print(f"T-03-8d: rc={rc} (expect 4 — baseline missing misroute.S1, candidate has it)")
assert rc == 4
print("Section 8 matrix demos done.")


[gates] config error -> exit 4: baseline missing gate row that candidate has: eval.gate.routing.misroute_negation.S1 (exit 4, T-03-8d)
T-03-8d: rc=4 (expect 4 — baseline missing misroute.S1, candidate has it)
Section 8 matrix demos done.


In [43]:
# Expected exit: 2 (T-03-8 — guardrail REVIEW(2): ALL gate rows equal, ONE guardrail
# present in BOTH and out of tolerance -> exit 2, NEVER touching gate verdict)
# Committed Mod-3 baselines carry NO guardrail rows (gate-only verdict scope, C4), so
# we synthesize a legacy-style baseline that DOES include one (per T-03-8 fixture).
b8_metrics = dict(baseline.metrics)
b8_metrics["eval.guardrail.position_bias.agreement"] = {
    "value": 1.0, "sample_size": 20, "computation": "stub"}
(scratch / "baseline_8_review.json").write_text(json.dumps({
    "id": "baseline-8-review", "created_utc": baseline.created_utc,
    "git_commit": baseline.git_commit, "schema_version": "1.0",
    "goldens_sha256": baseline.goldens_sha256, "corpus_sha256": baseline.corpus_sha256,
    "metrics": b8_metrics, "meta": baseline.meta}, indent=2, sort_keys=True) + "\n",
    encoding="utf-8")
cand8 = candidate_like_baseline()
cand8.append(_mv("eval.guardrail.position_bias.agreement", 0.80, 20))  # 1.0 -> 0.80, tol 0.03
rc = main(["--baseline", str(scratch / "baseline_8_review.json"),
           "--candidate", str(_write_candidate(cand8, "cand_08_review.json"))])
print(f"T-03-8 REVIEW: rc={rc} (expect 2 — gates all equal, guardrail regressed)")
assert rc == 2


[gates] verdict=2 regressed=['eval.guardrail.position_bias.agreement']
        detail eval.guardrail.position_bias.agreement: guardrail regressed: 0.8 vs baseline 1.0 tol 0.03 (T-03-8)
T-03-8 REVIEW: rc=2 (expect 2 — gates all equal, guardrail regressed)


In [44]:
# Expected exit: 0 (T-03-8c — faithful direction: guardrail row IS in baseline,
# candidate OMITS it -> skip + provenance note, never a verdict)
cand8c = candidate_like_baseline()                     # no guardrail appended
rc = main(["--baseline", str(scratch / "baseline_8_review.json"),
           "--candidate", str(_write_candidate(cand8c, "cand_08c_omit.json"))])
assert rc == 0
# the skip must be recorded in detail, not swallowed
v = compare(load_snapshot(scratch / "baseline_8_review.json"), cand8c)
skip = [d for k, d in v.detail.items() if "missing in candidate" in d]
assert skip, f"T-03-8c expected skip provenance note, detail={v.detail}"
print(f"T-03-8c guardrail-skip ok: rc={rc}, detail note verbatim: {skip[0]!r}")


[gates] verdict=0 regressed=none
        detail eval.guardrail.position_bias.agreement: guardrail missing in candidate — skipped, not verdict (T-03-8c)
T-03-8c guardrail-skip ok: rc=0, detail note verbatim: 'guardrail missing in candidate — skipped, not verdict (T-03-8c)'


In [45]:
# Expected exit: 1 (T-03-8i — coverage shrink: candidate S1 sample_size 33 !=
# registered expected 34 -> FAIL(1), detail names registered AND actual n; D39)
cand8i = candidate_like_baseline({"eval.gate.retriever.agreement.S1": (1.0, 33)})
rc = main(["--baseline", str(BASELINES_DIR / f"{BASELINE_ID}.json"),
           "--candidate", str(_write_candidate(cand8i, "cand_08i_coverage.json"))])
assert rc == 1
v = compare(baseline, cand8i)
assert "eval.gate.retriever.agreement.S1" in v.regressed
assert "33" in v.detail["eval.gate.retriever.agreement.S1"] and "34" in v.detail["eval.gate.retriever.agreement.S1"]
print(f"T-03-8i coverage ok: rc={rc}, detail: {v.detail['eval.gate.retriever.agreement.S1']}")


[gates] verdict=1 regressed=['eval.gate.retriever.agreement.S1']
        detail eval.gate.retriever.agreement.S1: coverage regression n=33 != registered 34 (T-03-8i)
T-03-8i coverage ok: rc=1, detail: coverage regression n=33 != registered 34 (T-03-8i)


In [46]:
# Expected exit: 4 (T-03-8e — zero baseline under RELATIVE tolerance -> ConfigError(4),
# applies to guardrail rows too (latency.p95 relative 0.20), D36/D10)
b8e_metrics = dict(baseline.metrics)
b8e_metrics["eval.guardrail.latency.p95"] = {"value": 0.0, "sample_size": 100, "computation": "stub"}
(scratch / "baseline_8e_zero.json").write_text(json.dumps({
    "id": "baseline-8e", "created_utc": baseline.created_utc,
    "git_commit": baseline.git_commit, "schema_version": "1.0",
    "goldens_sha256": baseline.goldens_sha256, "corpus_sha256": baseline.corpus_sha256,
    "metrics": b8e_metrics, "meta": baseline.meta}, indent=2, sort_keys=True) + "\n",
    encoding="utf-8")
cand8e = candidate_like_baseline()
cand8e.append(_mv("eval.guardrail.latency.p95", 0.1, 100))
rc = main(["--baseline", str(scratch / "baseline_8e_zero.json"),
           "--candidate", str(_write_candidate(cand8e, "cand_08e_zero.json"))])
print(f"T-03-8e zero-relative: rc={rc} (expect 4 — relative compare undefined at baseline 0.0)")
assert rc == 4
print("Section 8 matrix demos extended: T-03-8/8c/8i/8e now covered.")


[gates] config error -> exit 4: relative tolerance undefined at zero baseline: eval.guardrail.latency.p95 (exit 4, T-03-8e)
T-03-8e zero-relative: rc=4 (expect 4 — relative compare undefined at baseline 0.0)
Section 8 matrix demos extended: T-03-8/8c/8i/8e now covered.


Note: each demo cell's expected exit code is stated inline before running.

## Section 9 — Offline-seam self-check

Prove the module never imports live/judge code or touches env.

In [47]:
# --- Section 9: offline-seam self-check at the eval-module level ------------
# Cell 8 proved module hygiene at SETUP. Here we prove ENV-INDEPENDENCE and
# READ-ONLY determinism the way T-03-11b/11c do it (subprocess, stripped env),
# but at notebook scope: run the SAME eval code in a fresh, env-stripped,
# no-network child and require byte-identical canonical output.

_S9_SCRIPT = r"""
import hashlib, json, sys
from pathlib import Path
nb_path = sys.argv[1]
nb = json.loads(Path(nb_path).read_text())
src = "\n".join("".join(c["source"]) for c in nb["cells"][3:25] if c["cell_type"] == "code")
ns = {}
exec(compile(src, "<nb003-eval>", "exec"), ns)
report = ns["run_suite"](ns["GOLDENS_DIR"], ns["DOCS_DIR"])
print(hashlib.sha256(ns["compute_canonical_bytes"](report)).hexdigest())
"""

child = subprocess.run(
    [sys.executable, "-c", _S9_SCRIPT, str(ROOT / "jupyter_notebook" / "NB-003_regression_gates.ipynb")],
    capture_output=True, text=True, timeout=180,
    env={"PATH": "/usr/bin:/bin", "HOME": "X", "PYTHONPATH": str(ROOT)},
)
assert child.returncode == 0, f"child failed: {child.stderr[:500]}"
child_hash = child.stdout.strip().splitlines()[-1]
import hashlib
in_kernel_hash = hashlib.sha256(compute_canonical_bytes(report)).hexdigest()
assert child_hash == in_kernel_hash, f"env-stripped run differs: {child_hash} vs {in_kernel_hash}"
print(f"Env-independence ok: env-stripped/no-network child produced IDENTICAL canonical bytes.")
print(f"  sha256 = {child_hash[:16]}... (same as in-kernel)")


Env-independence ok: env-stripped/no-network child produced IDENTICAL canonical bytes.
  sha256 = 7a0c3896958fddec... (same as in-kernel)


In [48]:
# --- Deny-set closure scan (T-03-11 light) ----------------------------------
# Cell 8 checks the live session; here we scan OUR eval cells' IMPORTS for the
# live/judge deny-set. Raw-text contains the tokens by design (we must name them
# to deny them), so the scan targets `import`/`from` statements only — a probe
# of what the eval code COULD pull in, not bookkeeping strings.
import ast as _ast71
nb_src = ROOT / "jupyter_notebook" / "NB-003_regression_gates.ipynb"
nb_doc = json.loads(nb_src.read_text(encoding="utf-8"))
_DENY = ("config.judge", "judge_llm", "openai", "langchain", "httpx",
         "requests", "urllib.request", "socket")

def _eval_cell_imports() -> list[str]:
    hits = []
    for c in nb_doc["cells"]:
        if c["cell_type"] != "code":
            continue
        src = "".join(c["source"])
        try:
            tree = _ast71.parse(src)
        except SyntaxError:
            continue
        for node in _ast71.walk(tree):
            if isinstance(node, _ast71.Import):
                for a in node.names:
                    hits.append(a.name.split(".")[0])
            elif isinstance(node, _ast71.ImportFrom) and node.module:
                hits.append(node.module.split(".")[0])
    return hits

imported = _eval_cell_imports()
hits = [d for d in _DENY if d in imported]
assert not hits, f"deny-set module imported by eval cells: {hits}"
print(f"Deny-set closure scan ok: eval cells import only local/offline modules (n={len(imported)}).")
print("(Full T-03-11 closure + env-spy + read-only backstop: tests/test_gates.py.)")


Deny-set closure scan ok: eval cells import only local/offline modules (n=24).
(Full T-03-11 closure + env-spy + read-only backstop: tests/test_gates.py.)


Note: full enforcement (env spy, read-only/socket backstop, deny-set subset) lives in `tests/test_gates.py` (T-03-11/11b/11c/11d), not the notebook.

## Section 10 — Promotion checklist

Move finalized cells into `src/llmops/eval/{metric_registry,run_suite,snapshot,compare}.py` + `__init__.py` facade.

Checklist: registry validated; run_suite deterministic; snapshot round-trips; compare precedence matches 11-step spec; CLI exit codes verified.

Reminder: promoted code must additionally pass `tests/test_gates.py`, `ruff`, `mypy` (T-03-12) and the CI workflow contract (`llm_eval_gate.yml`) before merge.

## Section 11 — Verify block (mirrors task 03)

Reproduce the LLD's Verify commands post-promotion, from shell:

- `uv run python -m llmops.eval.compare --baseline eval/baselines/<id>.json --candidate <report>` + `echo $?` mapping (0..4)
- `uv run pytest tests/test_gates.py -q`
- `uv run ruff check .`
- `uv run mypy src/`

Closing note: this notebook is the scratchpad; `src/llmops/eval/` + `tests/test_gates.py` are the artifacts of record.